In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\felix\AppData\Local\Programs\Python\Python311\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\felix\AppData\Local\Programs\Python\Python311\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "c:\CSIE\My Projects\Data_Science\BoardGames_Analyzer\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\CSIE\My Projects\Data_Science\BoardGames_Analyzer\.venv\Lib\site-packag

ImportError: numpy.core.multiarray failed to import (auto-generated because you didn't call 'numpy.import_array()' after cimporting numpy; use '<void>numpy._import_array' to disable if you are certain you don't need it).

In [ ]:
PROCESSED = Path("../Dataset/Processed")
raw_dir = Path("../Dataset/Raw")

df_reviews = pd.read_csv(PROCESSED / "reviews_with_sentiment.csv", low_memory=False)
df_ratings = df_reviews[["user", "ID", "rating"]].dropna()
df_ratings = df_ratings.assign(rating=df_ratings["rating"].astype(float))
df_ratings.head()

In [ ]:
reader = Reader(rating_scale=(df_ratings.rating.min(), df_ratings.rating.max()))
data = Dataset.load_from_df(df_ratings[["user","ID","rating"]], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

algo = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
algo.fit(trainset)

predictions = algo.test(testset)
print("CF RMSE:", accuracy.rmse(predictions))
print("CF MAE:", accuracy.mae(predictions))

In [ ]:
all_items = df_ratings["ID"].unique().tolist()

def cf_scores_for_seed(seed_game_ids, user_id="__temp_user__"):
    global_mean = df_ratings["rating"].mean()
    seed_rating_map = {}
    for gid in seed_game_ids:
        seed_vals = df_ratings.loc[df_ratings["ID"]==gid, "rating"]
        seed_rating_map[gid] = seed_vals.mean() if len(seed_vals)>0 else global_mean

    sample_users = df_ratings[df_ratings["ID"].isin(seed_game_ids)]["user"].unique()
    if len(sample_users) > 5000:
        sample_users = np.random.choice(sample_users, 5000, replace=False)
    scores = {}
    for item in all_items:
        preds = []
        for u in sample_users[:200]:
            p = algo.predict(uid=u, iid=item).est
            preds.append(p)
        scores[item] = np.mean(preds) if preds else global_mean
    return pd.Series(scores)

In [ ]:
import joblib
joblib.dump(algo, "../Dataset/Processed/svd_cf_model.joblib")